### Automated Milling Workflow

In [ ]:
%load_ext autoreload
%autoreload 2

from odemis.acq.millmng import mill_patterns, MillingTaskSettings
from odemis.util.filename import make_unique_name
from pprint import pprint
from odemis import model
from odemis.acq.acqmng import acquire

from odemis.acq.stream import LiveStream, SEMStream, FIBStream, StaticSEMStream, FluoStream
import os
from pprint import pprint

In [ ]:
# connect to hardware

# stage
stage = model.getComponent(role="stage-bare")
print(f"Stage Position: {stage.position.value}")

# setup electron beam, det
electron_beam = model.getComponent(role="e-beam")
electron_det = model.getComponent(role="se-detector")
electron_focus = model.getComponent(role="ebeam-focus")

hwemtvas = set()
hwdetvas = set()

hwemt_vanames = ("beamCurrent", "accelVoltage", "resolution", "dwellTime", "horizontalFoV")
hwdet_vanames = ("brightness", "contrast", "detector_mode", "detector_type")
for vaname in model.getVAs(electron_beam):
    if vaname in hwemt_vanames:
        hwemtvas.add(vaname)
for vaname in model.getVAs(electron_det):
    if vaname in hwdet_vanames:
        hwdetvas.add(vaname)

sem_stream = SEMStream(
    name="SEM",
    detector=electron_det,
    dataflow=electron_det.data,
    emitter=electron_beam,
    focuser=electron_focus,
    hwemtvas=hwemtvas,
    hwdetvas=hwdetvas,
    blanker=None)

# setup ion beam, det
ion_beam = model.getComponent(role="ion-beam")
ion_det = model.getComponent(role="se-detector-ion")
ion_focus = model.getComponent(role="ion-focus")

hwemtvas = set()
hwdetvas = set()
# hwemt_vanames = ("beamCurrent", "accelVoltage", "resolution", "dwellTime", "horizontalFoV")
# hwdet_vanames = ("brightness", "contrast", "detector_mode", "detector_type")
for vaname in model.getVAs(ion_beam):
    if vaname in hwemt_vanames:
        hwemtvas.add(vaname)
for vaname in model.getVAs(ion_det):
    if vaname in hwdet_vanames:
        hwdetvas.add(vaname)

fib_stream = FIBStream(
    name="FIB",
    detector=ion_det,
    dataflow=ion_det.data,
    emitter=ion_beam,
    focuser=ion_focus,
    hwemtvas=hwemtvas,
    hwdetvas=hwdetvas,
)

print(f"SEM: {sem_stream}")
print(f"FIB: {fib_stream}")

# acquisitionStreams for FLM

ccd = model.getComponent(role="ccd")
light = model.getComponent(role="light")
focus = model.getComponent(role="focus")
em_filter = model.getComponent(role="filter")
fm_stream = FluoStream(
    name="FM",
    detector=ccd,
    dataflow=ccd.data,
    emitter=light,
    em_filter=em_filter,
    focuser=focus,
    # opm=self._main_data_model.opm,
    # detvas={"exposureTime"},
)

In [ ]:
stage1 = model.getComponent(role="stage")
print(stage1.getMetadata())
print(stage.getMetadata()[model.MD_SAMPLE_CENTERS])

In [ ]:
f = acquire([sem_stream, fib_stream])

data, err  = f.result()

sem_image, fib_image  = data

%matplotlib inline
from matplotlib import pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(sem_image, cmap="gray")
ax[1].imshow(fib_image, cmap="gray")
# ax[2].imshow(fm_image, cmap="gray")
plt.show()


In [ ]:
f = acquire([fm_stream])
data, err  = f.result()

fm_image = data

%matplotlib inline
from matplotlib import pyplot as plt
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.imshow(fm_image[0], cmap="gray")
plt.show()


In [ ]:
# for task_name, task in milling_tasks.items():
#     print(f"Task: {task_name}")
#     p = task.generate()
#     pprint(p)

In [ ]:
from odemis.acq.milling.feature import CryoLamellaFeature, CryoLamellaProject, create_new_project, create_new_feature
from odemis.acq.milling.tasks import load_milling_tasks, MILLING_ROUGH, MILLING_FINE, MILL_POLISHING
from typing import Dict


from datetime import datetime 
timestamp = datetime.timestamp(datetime.now())
# format as YYYY-MM-DD_HH-MM-SS
timestamp = datetime.fromtimestamp(timestamp).strftime('%Y-%m-%d_%H-%M-%S')
PROJECT_NAME = f"milling-project-{timestamp}"
PROJECT_PATH = "/home/patrick/development/scratch/project/automated-milling/"
os.makedirs(PROJECT_PATH, exist_ok=True)

# load milling tasks
task_list = [MILLING_ROUGH, MILLING_FINE, MILL_POLISHING]
milling_tasks = load_milling_tasks(os.path.join(PROJECT_PATH, "milling_tasks.yaml"), task_list)

# create project
project = create_new_project(PROJECT_PATH, PROJECT_NAME)


# features
# feature = CryoLamellaFeature(

create_new_feature(name="Lamella-1",
    position={"x": 50e-6, "y": 25e-6, "z": 15e-6, "rx": 0, "rz": 0},
    milling_tasks=milling_tasks,
    project=project,
)

create_new_feature(name="Lamella-1",
    position={"x": 33e-6, "y": 100e-6, "z": 66e-6, "rx": 0, "rz": 0},
    milling_tasks=milling_tasks,
    project=project,
)

pprint(project.features)

# steps
# 1. create feature
# 2. select position
# 3. select milling tasks
# 4. select alignment
# ready to mill


# TODO: ref image data -> when to acquire reference image
# TODO: save directory -> project directory / feature directory



In [ ]:
print(project.features.keys())

In [ ]:

f = stage.moveAbs({"x": 0, "y": 0, "z": 0, "rx": 0, "rz": 0})
f.result()


In [ ]:
# loop through each position, mill lamella
from odemis.acq.align.shift import MeasureShift
from odemis.dataio import find_fittest_converter

_exporter = find_fittest_converter("filename.ome.tiff")

# TODO: set reference image correctly
for k, f in project.features.items():
    f.reference_image = fib_image

import numpy as np



# TODO: insert fm acquisition between milling tasks

for task_num, task_name in enumerate(task_list, 1):
    print(f"Starting {task_name} for {len(project.features)} features...")
    task_num = f"{task_num:02d}"
    for name, feature in project.features.items():

        feature_name = feature.name.value
        print(f"Feature: {feature.name.value}")
        print(f"Position: {feature.position.value}")
        print(f"Previous Status: {feature.status.value}")
        print("--" * 10)
        print(f"Starting {task_name} for {feature.name.value}")

        # move to position
        f = stage.moveAbs(feature.position.value) # TODO: make this move safer using posture manager
        f.result()

        # reset beam shift
        ion_beam.shift.value = (0, 0)

        # beam shift alignment
        f = acquire([fib_stream])
        data, _ = f.result()
        new_image = data[0]

        # roll data by a random amount
        import random
        x, y = random.randint(0, 100), random.randint(0, 100)
        new_image = np.roll(new_image, [x, y], axis=[0, 1])
        print(f"Shifted image by {x}, {y} pixels")

        align_filename = os.path.join(feature.path, f"{feature_name}-{task_num}-{task_name}-Alignment-FIB.ome.tiff").replace(" ", "-") # TODO: make unique
        _exporter.export(align_filename, new_image)       

        ref_image = feature.reference_image # load from directory?
        def align_reference_image(ref_image, new_image, scanner):
            shift_px = MeasureShift(ref_image, new_image, 10)
            # shift_px = (1, 1)
            pixelsize = ref_image.metadata[model.MD_PIXEL_SIZE]
            shift_m = (shift_px[0] * pixelsize[0], shift_px[1] * pixelsize[1])

            previous_shift = scanner.shift.value
            print(f"Previous: {previous_shift}, Shift: {shift_m}")
            shift = (shift_m[0] + previous_shift[0], shift_m[1] + previous_shift[1])  # m
            scanner.shift.value = shift
            print(f"Shift: {scanner.shift.value}")

        align_reference_image(ref_image, new_image, scanner=ion_beam)

        # mill patterns
        task = feature.milling_tasks[task_name]
        pprint(task)
        f = mill_patterns(task)
        f.result()

        # acquire images
        f = acquire([sem_stream, fib_stream])
        data, ex = f.result()
        sem_image, fib_image = data

        # save images
        sem_filename = os.path.join(feature.path, f"{feature_name}-{task_num}-{task_name}-Finished-SEM.ome.tiff").replace(" ", "-") # TODO: make unique 
        fib_filename = os.path.join(feature.path, f"{feature_name}-{task_num}-{task_name}-Finished-FIB.ome.tiff").replace(" ", "-") # TODO: make unique
        _exporter.export(sem_filename, sem_image)
        _exporter.export(fib_filename, fib_image)       


        # move to flm position
        # TODO: use pm to move to flm position
        print(f"Moving to FLM position for {feature.name.value}")
        # set objective position
        f = focus.moveAbs({"z": feature.focus_position.value})
        f.result()

        # TODO: get fm acquisition settings ?

        # acquire fm z-stack
        f = acquire([fm_stream])
        data, ex = f.result()
        fm_image = data
        # save flm image
        fm_filename = os.path.join(feature.path, f"{feature_name}-{task_num}-{task_name}-Finished-FLM.ome.tiff").replace(" ", "-")
        _exporter.export(fm_filename, fm_image)

        # plot images
        fig, ax = plt.subplots(1, 3, figsize=(10, 5))
        ax[0].imshow(sem_image, cmap="gray")
        ax[1].imshow(fib_image, cmap="gray")
        ax[2].imshow(fm_image[0], cmap="gray")
        plt.show()

        # update status
        feature.status.value = task_name

        print(f"Finished {task_name} for {feature.name.value}")

        # save project
        project.save()

    print("-"*50)


# OVERVIEW Acquisitions

In [ ]:
from odemis.acq.stitching import get_tiled_areas, get_zstack_levels, acquireTiledArea, acquireOverview, FocusingMethod, REGISTER_IDENTITY, WEAVER_MEAN
from odemis.util.filename import create_filename
from odemis.dataio import find_fittest_converter

sample_centers_raw = stage.getMetadata()[model.MD_SAMPLE_CENTERS]
sample_centers = {n: (p["x"], p["y"]) for n, p in sample_centers_raw.items()}

# move to base
# stage.moveAbs({"x": 0, "y": 0, "z": 0, "rx": 0, "rz": 0}).result()


overlap = 0.1
autofocus = False

electron_beam.horizontalFoV.value = 500e-6
acq_streams = [sem_stream]

print(sample_centers)
areas = get_tiled_areas(
    pos=stage.position.value,
    streams=[sem_stream],
    tiles_nx=2,
    tiles_ny=2,
    overlap=0.1,
    tiling_rng={"x": [-1e-3, 1e-3], "y": [-1e-3, 1e-3]},
    selected_grids=["GRID 1", "GRID 2"],
    sample_centers=sample_centers,
    whole_grid=True,
)

print(areas)

focus_mtd = FocusingMethod.NONE

save_dir = os.path.join("/home/patrick/development/scratch", f"acq/grids")
os.makedirs(save_dir, exist_ok=True)
filename = create_filename(save_dir, "{datelng}-{timelng}-overview",
                                        ".ome.tiff")
print(f"Filename: {filename}")
assert filename.endswith(".ome.tiff")
dirname, basename = os.path.split(filename)
tiles_dir = os.path.join(dirname, basename[:-len(".ome.tiff")] + "-tiles")
filename_tiles = os.path.join(tiles_dir, basename)
os.makedirs(tiles_dir, exist_ok=True)

print(filename_tiles)
print(f"Acquiring ...")


acq_future = acquireOverview(
                streams=acq_streams,              # restrict to a single stream if using autofocus (sem, fib)
                stage=stage, 
                areas=areas, 
                focus=electron_focus,             # replace with sem-focus, fib-focus
                detector=electron_det,            # replace with detector
                overlap=overlap, 
                settings_obs=None, 
                log_path=filename_tiles, 
                zlevels=None,
                registrar=REGISTER_IDENTITY, 
                weaver=WEAVER_MEAN, 
                focusing_method=FocusingMethod.NONE)

data = acq_future.result()

exporter = find_fittest_converter(filename)
exporter.export(filename, data)

# das = []
# grid_names = ["grid-01", "grid-02"]
# for grid_name, area in zip(grid_names, areas):

#     save_dir = os.path.join("/home/patrick/development/scratch", f"acq/{grid_name}")
#     os.makedirs(save_dir, exist_ok=True)
#     filename = create_filename(save_dir, "{datelng}-{timelng}-overview",
#                                             ".ome.tiff")
#     print(f"Filename: {filename}")
#     assert filename.endswith(".ome.tiff")
#     dirname, basename = os.path.split(filename)
#     tiles_dir = os.path.join(dirname, basename[:-len(".ome.tiff")] + "-tiles")
#     filename_tiles = os.path.join(tiles_dir, basename)
#     os.makedirs(tiles_dir, exist_ok=True)

#     print(filename_tiles)
#     print(f"Acquiring {grid_name}...")

#     acq_future = acquireTiledArea(streams=acq_streams, 
#                                 stage=stage, 
#                                 area=area,
#                                 overlap=overlap,
#                                 settings_obs=None,
#                                 zlevels=None,
#                                 log_path=filename_tiles,
#                                 weaver=WEAVER_MEAN,
#                                 registrar=REGISTER_IDENTITY,
#                                 )


#     data = acq_future.result()
#     das.append(data)

#     # save
#     exporter = find_fittest_converter(filename)
#     exporter.export(filename, data)


In [ ]:
import matplotlib.pyplot as plt
for da in das:

    plt.imshow(da[0], cmap="gray")
    plt.show()

In [ ]:
data = acq_future.result(1)  # timeout is just for safety

In [ ]:
pprint(data[0].metadata)

In [ ]:
fib_stream.focuser.axes["z"].range

In [ ]:
focus_points = None
import numpy

FOCUS_RANGE_MARGIN = 100e-6  # m
_focus_stream = next((sd for sd in acq_streams if sd.focuser is not None), None)
_good_focus = _focus_stream.focuser.position.value['z']
focuser_range = _focus_stream.focuser.axes['z'].range
# Calculate the focus range by half the focus margin on each side of good focus
focus_rng = (_good_focus - FOCUS_RANGE_MARGIN / 2, _good_focus + FOCUS_RANGE_MARGIN / 2)
# Clip values with focuser_range
_focus_rng = (max(focus_rng[0], focuser_range[0]), min((focus_rng[1], focuser_range[1])))
print("Calculated focus range ={}".format(_focus_rng))

# used in re-focusing method
_focus_points = numpy.array(focus_points) if focus_points else None
# triangulate focus points
_tri_focus_points = None
if focus_points is not None:
    if len(focus_points) >= 3:
        self._tri_focus_points = Delaunay(self._focus_points[:, :2])
    elif len(focus_points) == 1:
        # Triangulation needs minimum three points to define a plane
        # When the number of focus points is less than three
        # The focus is set constant and based on a single focus point
        pass
    else:
        raise ValueError(f"focus_points length {len(focus_points)} is not supported")

In [ ]:
from odemis.util.dataio import open_acquisition
import matplotlib.pyplot as plt

image = open_acquisition("/home/patrick/development/scratch/acq/grids/20240611-183438-overview.ome.tiff")

plt.imshow(image[0], cmap="gray")
plt.show()